# <span style="font-weight:bold;color:blue;font-size:1.25em">Reviewing $t$-Tests, Regression, and ANOVA</span> 

This notebook will review three key statistical topics:

1. Independent Samples $t$-Test
2. Linear Regression
3. ANOVA

In [1]:
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import seaborn as sns
import scipy.stats as stats
from scipy.stats import gaussian_kde

In [2]:
def describe(data, var = None):
  if isinstance(data, Table):
    data = data.column(var)
  names = np.array(["mean", "std", "n", " : : ", "min", "Q1", "median", "Q3", "max", "IQR"])
  values = np.array([round(np.mean(data),1), round(stats.tstd(data),2), len(data) - 1, "", min(data), np.percentile(data, 25), np.median(data), np.percentile(data, 75), max(data), stats.iqr(data)])
  tab = Table().with_columns('Names', names, 'Values', values)  ;  transposed_tab = Table().with_columns( zip(tab.column('Names'), tab.column('Values')) ).show()
  return

def describes(data, num, group):
  dat = data.column(num)  ;  group_var = data.column(group)  ;  groups = np.unique(group_var)  ;  rows = []  ;  tab = Table(["mean", "std", "n", " : : ", "min", "Q1", "median", "Q3", "max", "IQR"])
  for g in groups:
    # Filter data for the current group
    sub_dat = dat[group_var == g]  ;    rows = [g, round(np.mean(sub_dat), 1), round(stats.tstd(sub_dat), 2), len(sub_dat) - 1, min(sub_dat), np.percentile(sub_dat, 25), np.median(sub_dat), np.percentile(sub_dat, 75), max(sub_dat), stats.iqr(sub_dat)]        
    tab = tab.with_row(rows)
  tab.show()
  return 

def box(data, num = None, sm = False):
  if isinstance(data, Table):
    df = data.to_df()  ;  sns.boxplot(data=df, y=num, width=0.45, showmeans=sm)  ;
    plots.title(f'Boxplot of {num}')  ;  plots.xticks([], [])  ;  plots.show()  
  else:
    if num == True:
      sm = True  ;  num = 'Variable'
    if num == None:
      num = 'Variable'
    plots.figure(figsize=(6, 6))  ;  plots.boxplot(data, widths = 0.45, showmeans=sm)
    plots.title(f'Boxplot of {num}')  ;  plots.xticks([], [])  ;  plots.show()
  return

def boxes(data, num, cat, sm = False):
  if not isinstance(data, Table):
    print("ERROR: First input must be table name.")
  else:
    df = data.to_df()  ;  sns.boxplot(data=data, x=cat, y=num, width=0.75, showmeans=sm)
    plots.title(f'Boxplot of {num} Grouped by {cat}')  ;  plots.show()
  return

def density(data, num = None):
  if isinstance(data, Table):
    sns.kdeplot( data = data.column(num), bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
  else:
    if num == None:
      num = 'Variable'  ;  sns.kdeplot( data = data, bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
    else:
      sns.kdeplot( data = data, bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
  return

def densities(tab, var, group_var):
  groups = tab.column(group_var) ;  unique_groups = np.unique(groups)  ;  plots.figure()
  for g in unique_groups:
    # Filter data for the current group
    mask = (groups == g)
    x = tab.column(var)[mask]   
    # Calculate KDE for the group subset
    kde = gaussian_kde(x)
    x_grid = np.linspace(x.min() - 1, x.max() + 1, 512)  ;    plots.plot(x_grid, kde(x_grid), label=str(g))   
  plots.title(f'Density Plot of {var} by {group_var}')  ;  plots.xlabel(f'{var}')  ;  plots.ylabel('Density')
  plots.legend()  ;  plots.show()
  return

## <span style="font-weight:bold;color:blue;font-size:1.25em">Example 1: Monthly Gear Spending by Outdoor Hobby Preference</span>

This example provides research evaluating whether college students' **monthly spending on outdoor gear** differs between those whose primary hobby is an **Aquatic Outdoor Activity** (Kayaking, Canoeing, Fishing) versus a **Terrestrial Outdoor Activity** (Hiking, Camping, Hunting).

An outdoor recreational club surveys $30$ college students to determine if students primary engaged in water-based hobbies spend significantly more money per month on gear than those engaged in land-based hobbies. Determine at the $\alpha = 0.05$ level whether spending on gear depends upon type of outdoor activity.

In [3]:
data2 = make_array(62,78,50,85,68,55,72,90,60,82,48,75,65,70,58,38,45,28,52,40,32,48,60,35,55,30,42,50,36,25)
labels2 = make_array('Acquatic','Acquatic','Acquatic','Acquatic','Acquatic','Acquatic','Acquatic','Acquatic','Acquatic','Acquatic','Acquatic','Acquatic','Acquatic','Acquatic','Acquatic','Terrestrial','Terrestrial','Terrestrial','Terrestrial','Terrestrial','Terrestrial','Terrestrial','Terrestrial','Terrestrial','Terrestrial','Terrestrial','Terrestrial','Terrestrial','Terrestrial','Terrestrial')    

gear_spend = Table().with_columns('Outdoor Hobby', labels2, 'Gear Spending (monthly)', data2)
gear_spend.show(7)

Outdoor Hobby,Gear Spending (monthly)
Acquatic,62
Acquatic,78
Acquatic,50
Acquatic,85
Acquatic,68
Acquatic,55
Acquatic,72


### Hypothesis

### Data Checks

### Run Test

### Reporting Out

## <span style="font-weight:bold;color:blue;font-size:1.25em">Example 2: College Student Participation in Outdoor Activities</span>

A campus recreation researcher surveys 20 college students to determine if their **weekly exercise hours** differ significantly depending on their primary outdoor hobby. Determine at the $\alpha=0.05$ level whether exericse hours varry based upon preferred outdoor hobby.

---
### Hypothesis
- $H_0 :$ All groups have the same mean weekly exercise hours.
- $H_a :$ At least one outdoor activity group has a different mean weekly exercise time.

In [4]:
data = make_array(6.5, 8,7,9,7.5, 4,5.5,4.5,6,5,7.5,6,8.5,7,9,3,4.5,2.5,5,3.5)
labels = make_array('Hiking','Hiking','Hiking','Hiking','Hiking','Camping','Camping','Camping','Camping','Camping','Kayaking','Kayaking','Kayaking','Kayaking','Kayaking','Fishing','Fishing','Fishing','Fishing','Fishing')
exercise = Table().with_columns('Primary Hobby',labels,'Exercise Hours',data)
exercise.show(7)

Primary Hobby,Exercise Hours
Hiking,6.5
Hiking,8
Hiking,7
Hiking,9
Hiking,7.5
Camping,4
Camping,5.5


### Data Checks


### Run Test

### Reporting Out

## <span style="font-weight:bold;color:blue;font-size:1.25em">Example 3: Days Outdoors vs. Gear Spending</span>

A recreation researcher surveys $N = 20$ college students involved in outdoor clubs. They record (1) the number of **days spent outdoors** per year and the  **Annual gear spending** (in dollars) for each student.

---

### Resarch Question
Determine at the $\alpha = 0.05$ whether a significant correlation exists between outdoor days and annual gear spending.

---

### Hypothesis

...

In [5]:
days_outdoors = make_array(12,15,18,22,25,28,30,35,38,40,42,45,50,52,58,60,65,70,75,80)
annual_spending = make_array(180,320,210,410,350,290,480,390,560,310,620,440,520,380,650,490,710,540,820,600)

daysout_vs_spend = Table().with_columns('Days Spent Outdoors',days_outdoors,'Annual Spending on Gear (yearly)',annual_spending)
daysout_vs_spend.show(7)

Days Spent Outdoors,Annual Spending on Gear (yearly)
12,180
15,320
18,210
22,410
25,350
28,290
30,480


### Data Checks

### Run Test

### Reporting Out